# Part I. Pre-processing event data

#### Import Python packages 

In [1]:
import \
    cassandra, \
    csv, \
    glob, \
    json, \
    numpy as np, \
    os, \
    pandas as pd, \
    re

from cassandra.cluster import Cluster

#### Create list of filepaths to process event data

In [2]:
# variables
data_file = 'event_datafile_new.csv'
data_sub_dir = '/event_data'

In [3]:
def retrieveFilePathList(data_sub_dir):
    """
    This fct. scans data_sub_dir and 
    returns a collection of relative file paths with file name.
    """
    filepath = os.getcwd() + data_sub_dir

    for root, dirs, files in os.walk(filepath):
        # join the file path and roots with the subdirectories
        file_path_list = glob.glob(os.path.join(root,'*'))
    
    return file_path_list

#### Merging data into a master data file

In [4]:
def buildDataFile(file_path_list, data_file):
    """
    This fct. composes records from file addresses in the file_path_list and
    writes records into data_file.
    """
    
    full_data_rows_list = [] 

    for f in file_path_list:

        # reading event data 
        with open(f, 'r', encoding = 'utf8', newline='') as csvfile: 
            csvreader = csv.reader(csvfile)
            # skip header
            next(csvreader)
            # collect rows
            for line in csvreader:
                #print(line)
                full_data_rows_list.append(line)

    # set csv formatting for Apache Cassandra 
    csv.register_dialect('myDialect', quoting=csv.QUOTE_ALL, skipinitialspace=True)

    # write data into data_file
    with open(data_file, 'w', encoding = 'utf8', newline='') as f:
        writer = csv.writer(f, dialect='myDialect')
        writer.writerow([ \
                    'artist','firstName','gender', \
                    'itemInSession','lastName','length', \
                    'level','location','sessionId', \
                    'song','userId' \
                    ])
        for row in full_data_rows_list:
            if (row[0] == ''):
                continue
            writer.writerow(( \
                    row[0], row[2], row[3], \
                    row[4], row[5], row[6], \
                    row[7], row[8], row[12], \
                    row[13], row[16] \
                    ))       


#### Auxiliary functions

In [5]:
def checkRecordNumber(data_file):
    """
    This fct. counts the number of rows in data_file.
    
    Returns: row_count (int)
    """
    with open(data_file, 'r', encoding = 'utf8') as f:
        row_count =  sum(1 for line in f)
    return row_count

def checkFile(data_file):
    """
    This fct. checks if data_file exists and 
    if there are any records in it.
    
    Returns: Boolean
    """
    
    if (data_file not in os.listdir() 
            or checkRecordNumber(data_file) != 6821):
        value = True
    else:
        value = False
    return value

def instantiateDF(file):
    global df
    df = pd.read_csv(file, encoding = 'utf8')

#### Read 'data_file' an instantiate data frame

In [6]:
if checkFile(data_file):
    file_path_list = retrieveFilePathList(data_sub_dir)
    buildDataFile(file_path_list, data_file) 
else:
    pass
instantiateDF(data_file)

# Part II. Load data into Apache Cassandra and run queries

The source file, event_datafile_new.csv, contains the following columns: 
1. artist 
2. first name of user
3. gender of user
4. item number in session
5. last name of user
6. length of the song
7. level (paid or free song)
8. location of the user
9. session-ID
10. song title
11. user-ID

Denormalized source-data instance:

<img src="images/image_event_datafile_new.jpg">

## A - Initialize Cassandra cluster and keyspace

#### Creating a cluster

In [7]:
# connect to the Cassandra local host on 127.0.0.1
cluster = Cluster(['127.0.0.1'])

# initialize session
session = cluster.connect()

#### Create and set Keyspace

In [8]:
try:
    # create keyspace 
    session.execute(
        """
        CREATE KEYSPACE IF NOT EXISTS sparkify 
        WITH REPLICATION = 
        { 'class' : 'SimpleStrategy', 'replication_factor' : 1 }
        """
        )
    # set keyspace
    session.set_keyspace('sparkify')
except Exception as e:
    print(e)

## B - Define queries

### B.1 - Give me the artist, song title and song's length in the music app history that was heard during  sessionId = 338, and itemInSession  = 4

```
SELECT
    t1.artist,
    t1.song,
    t1.length
FROM tableOne t1
WHERE 
    t1.sessionId = 338
AND t1.itemInSession = 4

```   

### B.2 - Give me only the following: name of artist, song (sorted by itemInSession) and user (first and last name) for userid = 10, sessionid = 182


```
SELECT
    t2.artist,
    t2.song,
    t2.firstName,
    t2.lastName
FROM tableTwo t2
WHERE 
    t2.userId = 10
AND t2.sessionId = 182
SORT BY t2.itemInSession
```   

### B.3 - Give me every user name (first and last) in my music app history who listened to the song 'All Hands Against His Own'

```
SELECT
    t3.firstName,
    t3.lastName
FROM tableThree t3
WHERE
    t3.song like 'All Hands Against His Own'
```   

## C - Preliminary Exploratory Data Analysis

In [9]:
df.head()

,artist,firstName,gender,itemInSession,lastName,length,level,location,sessionId,song,userId
0,Barry Tuckwell/Academy of St Martin-in-the-Fie...,Mohammad,M,0,Rodriguez,277.15873,paid,"Sacramento--Roseville--Arden-Arcade, CA",961,Horn Concerto No. 4 in E flat K495: II. Romanc...,88
1,Jimi Hendrix,Mohammad,M,1,Rodriguez,239.82975,paid,"Sacramento--Roseville--Arden-Arcade, CA",961,Woodstock Inprovisation,88
2,Building 429,Mohammad,M,2,Rodriguez,300.61669,paid,"Sacramento--Roseville--Arden-Arcade, CA",961,Majesty (LP Version),88
3,The B-52's,Gianna,F,0,Jones,321.54077,free,"New York-Newark-Jersey City, NY-NJ-PA",107,Love Shack,38
4,Die Mooskirchner,Gianna,F,1,Jones,169.29914,free,"New York-Newark-Jersey City, NY-NJ-PA",107,Frisch und g'sund,38


#### Focus on record uniqueness and relation between session-ID, items and user-ID

In [10]:
len(df[df['sessionId'].duplicated()])

6044

In [11]:
len(df[df[['userId', 'sessionId']].duplicated()])

6044

In [12]:
len(df[df[['userId', 'itemInSession']].duplicated()])

4967

In [13]:
len(df[df[['sessionId', 'itemInSession']].duplicated()])

0

It looks like `sessionId` and `itemInSession`suffice as unique identifier.

In [14]:
df[[
    'userId', 'sessionId','itemInSession'
    ]][
        df[[
            'userId', 'sessionId', 'itemInSession'
            ]].duplicated()==True
        ]

,userId,sessionId,itemInSession


Adding user-ID maintains uniqueness.

#### Checking session data

In [15]:
df.query('(sessionId == 338)')# & (itemInSession == 4)')

,artist,firstName,gender,itemInSession,lastName,length,level,location,sessionId,song,userId
441,Pixies,Ava,F,1,Robinson,89.36444,free,"New Haven-Milford, CT",338,Build High,50
442,The Roots / Jack Davey,Ava,F,2,Robinson,155.95057,free,"New Haven-Milford, CT",338,Atonement,50
443,Mike And The Mechanics,Ava,F,3,Robinson,275.12118,free,"New Haven-Milford, CT",338,A Beggar On A Beach Of Gold,50
444,Faithless,Ava,F,4,Robinson,495.30730,free,"New Haven-Milford, CT",338,Music Matters (Mark Knight Dub),50


#### Checking relation of artist and song to session-ID and item

In [16]:
df[[
    'artist', 'song', 'length', 
    'sessionId', 'itemInSession'
    ]][
        df[[
            'artist', 'song', 'length', 
            'sessionId', 'itemInSession'
            ]].duplicated()==True
        ]

,artist,song,length,sessionId,itemInSession


#### Records for song 'All Hands Against His Own'

In [17]:
df.query('song == "All Hands Against His Own"')

,artist,firstName,gender,itemInSession,lastName,length,level,location,sessionId,song,userId
2792,The Black Keys,Tegan,F,25,Levine,196.91057,paid,"Portland-South Portland, ME",611,All Hands Against His Own,80
5135,The Black Keys,Sara,F,31,Johnson,196.91057,paid,"Winston-Salem, NC",152,All Hands Against His Own,95
6298,The Black Keys,Jacqueline,F,50,Lynch,196.91057,paid,"Atlanta-Sandy Springs-Roswell, GA",559,All Hands Against His Own,29


## D - Table creation and data load

#### Table names and primary keys

1.**Table 1**
    
   * **Name**: table provides artist, song, length for item(s) in session 
    &rarr; `songs_by_session`;
   * **Primary Key**: combination of session-ID and item No;
   * **Partition Key**: session-ID.


2.**Table 2**
    
   * **Name**: table provides user data and song data for user and session, and item(s), eventually &rarr; `user_and_songs_by_userid_and_session`;
   * **Primary Key**: combination of user-ID, session-ID and item No;
   * **Partition Key**: combination of user-ID and session-ID for performance purpose.

3.**Table 3**
    
   * **Name**: table provides user data by song title &rarr; `users_by_song_title`;
   * **Primary Key**: combination of song and user-ID, for the latter works as single identifier for user data;
   * **Partition Key**: song.

#### Table creation

In [18]:
query_t1 = """
        CREATE TABLE IF NOT EXISTS songs_by_session ( \
            sessionId int, \
            itemInSession int, \
            artist varchar, \
            song varchar, \
            length float, \
            PRIMARY KEY ((sessionId), itemInSession) \
            ); \
"""

query_t2 = """
        CREATE TABLE IF NOT EXISTS user_and_songs_by_userid_and_session ( \
            userId int, \
            sessionId int, \
            itemInSession int, \
            lastName varchar, \
            firstName varchar, \
            artist varchar, \
            song varchar, \
            PRIMARY KEY ((userId, sessionId), itemInSession) \
            ); \
"""

query_t3 = """
        CREATE TABLE IF NOT EXISTS users_by_song_title ( \
            song varchar, \
            userId int, \
            lastName varchar, \
            firstName varchar, \
            PRIMARY KEY ((song), userId) \
            ); \
"""

queries = [query_t1, query_t2, query_t3]

for query in queries:    
    try:
        session.execute(query)
    except Exception as e:
        print(e)

#Ref.: https://cassandra.apache.org/doc/latest/cql/types.html

#### Data load

In [19]:
query_i1 = """
        INSERT INTO songs_by_session ( \
            sessionId, \
            itemInSession, \
            artist, \
            song, \
            length \
            ) \
        VALUES ( \
            %s, %s, %s, \
            %s, %s \
            ); \
"""

query_i2 = """
        INSERT INTO user_and_songs_by_userid_and_session ( \
            userId, \
            sessionId, \
            itemInSession, \
            lastName, \
            firstName, \
            artist, \
            song \
            ) \
        VALUES ( \
            %s, %s, %s, \
            %s, %s, %s, \
            %s \
            ); \
"""

query_i3 = """
        INSERT INTO users_by_song_title ( \
            song, \
            userId, \
            lastName, \
            firstName \
            ) \
        VALUES ( \
            %s, %s, \
            %s, %s \
            ); \
"""

with open(data_file, encoding = 'utf8') as f:
    csvreader = csv.reader(f)
    next(csvreader)
    for line in csvreader:
        session.execute(query_i1, (
                                    int(line[8]), int(line[3]),
                                    line[0], line[9], float(line[5]) 
                                    )
                        )
        session.execute(query_i2, (
                                    int(line[10]), int(line[8]), 
                                    int(line[3]), line[4], 
                                    line[1], line[0], line[9]
                                    )
                        )
        session.execute(query_i3, (
                                    line[9], int(line[10]),
                                    line[4], line[1]
                                    )
                        )
        

## C - Test

#### Querying tables according to the requirements

In [20]:
# wrapper for pandas DF from iterable and column names
def pandas_factory(colnames, rows):
    return pd.DataFrame(rows, columns=colnames)

def run_(query, wrapper=pandas_factory):
    global session
    try:
        # passing wrapper
        session.row_factory = wrapper
        # executing query
        result = session.execute(query)
        # extracting DF from ResultSet
        df = result._current_rows
        return df
    except Exception as e:
        print(e)

# Ref.s:
# 1 - https://www.datacamp.com/community/tutorials/role-underscore-python
# 2 - https://thispointer.com/python-how-to-use-global-variables-in-a-function/
# 3 - https://stackoverflow.com/questions/41247345/python-read-cassandra-data-into-pandas

In [21]:
query1 = """
        SELECT \
            artist, \
            song, \
            length \
        FROM songs_by_session \
        WHERE  \
            sessionId = 338; \
"""

run_(query1)

,artist,song,length
0,Pixies,Build High,89.364441
1,The Roots / Jack Davey,Atonement,155.950577
2,Mike And The Mechanics,A Beggar On A Beach Of Gold,275.121185
3,Faithless,Music Matters (Mark Knight Dub),495.307312


In [22]:
query2 = """
        SELECT \
            lastName, \
            firstName, \
            artist, \
            song \
        FROM user_and_songs_by_userid_and_session \
        WHERE \
            userId = 10 \
        AND sessionId = 182; \
"""

run_(query2)

,lastname,firstname,artist,song
0,Cruz,Sylvie,Down To The Bone,Keep On Keepin' On
1,Cruz,Sylvie,Three Drives,Greece 2000
2,Cruz,Sylvie,Sebastien Tellier,Kilometer
3,Cruz,Sylvie,Lonnie Gordon,Catch You Baby (Steve Pitron & Max Sanna Radio...


In [23]:
query3 = """
        SELECT \
            lastName, \
            firstName \
        FROM users_by_song_title \
        WHERE \
            song = 'All Hands Against His Own'; \
"""

run_(query3)

,lastname,firstname
0,Lynch,Jacqueline
1,Levine,Tegan
2,Johnson,Sara


### Drop tables

In [24]:
tables = ['songs_by_session', 'user_and_songs_by_userid_and_session', 'users_by_song_title']
for table in tables:
    query = "DROP TABLE IF EXISTS {};".format(table)
    try:
        rows = session.execute(query)
    except Exception as e:
        print(e)

### Close the session and cluster connection

In [25]:
session.shutdown()
cluster.shutdown()